### Trabalho Fase 1 do Curso de Pós-Graduação FIAP IA para Devs

#### Parte 6 - Classificação supervisionada: detecção de indícios de câncer de mama

Este notebook implementa a etapa de classificação supervisionada usando a base tratada do DATASUS (SIH/SUS RJ 2025).
O objetivo é criar um target binário a partir de `CANCER_MAMA_NIVEL > 0`, preparar os dados com `Pipeline` e `ColumnTransformer` e comparar um baseline ingênuo, uma regressão logística e um KNN.

## Sumário da Parte 6

#### Item 1 - Definição do target
#### Item 2 - Preparação dos dados
#### Item 3 - Separação entre treino e teste
#### Item 4 - Pipeline e treinamento dos modelos
#### Item 5 - Avaliação dos modelos
#### Item 6 - Comparação dos resultados
#### Item 7 - Validação cruzada
#### Item 8 - ROC e AUC
#### Conclusão da Parte 6


#### Item 1 - Definição do target

Vamos criar a variável binária `TARGET`: 0 significa sem indício e 1 significa com indício de câncer de mama.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')

candidate_paths = [
    Path('2025_tratado.parquet'),
    Path('ia_tech_challenge_01/2025_tratado.parquet'),
    Path('/content/2025_tratado.parquet'),
    Path('/content/drive/MyDrive/2025_tratado.parquet'),
    Path('/content/drive/MyDrive/ia_tech_challenge_01/2025_tratado.parquet'),
]

path = next((candidate for candidate in candidate_paths if candidate.exists()), None)
if path is None:
    searched = '\n'.join(f'- {candidate}' for candidate in candidate_paths)
    raise FileNotFoundError(
        'Arquivo de base tratada não encontrado. Verifique onde o arquivo '
        "'2025_tratado.parquet' foi salvo e ajuste o caminho na célula.\n"
        f'Caminhos verificados:\n{searched}'
    )

print(f'Arquivo localizado em: {path.resolve()}')
dados = pd.read_parquet(path)
TARGET = 'TARGET'
dados[TARGET] = (dados['CANCER_MAMA_NIVEL'] > 0).astype(int)

print('Dimensões da base tratada:', dados.shape)
print('Distribuição do target:')
print(dados[TARGET].value_counts().sort_index())
print('\nProporção do target:')
print(dados[TARGET].value_counts(normalize=True).sort_index().rename('proporcao'))

target_plot = dados[TARGET].value_counts().sort_index().rename_axis('Classe').reset_index(name='Quantidade')
target_plot['Classe'] = target_plot['Classe'].map({0: 'Sem indicio', 1: 'Com indicio'})

plt.figure(figsize=(6, 4))
ax = sns.barplot(data=target_plot, x='Classe', y='Quantidade', hue='Classe', palette='Blues', legend=False)
ax.set_title('Distribuicao do target')
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
for patch in ax.patches:
    ax.annotate(f"{int(patch.get_height())}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


#### Item 2 - Preparação dos dados

A seleção de variáveis prioriza atributos demográficos, administrativos e de utilização hospitalar.
Variáveis diretamente derivadas de diagnóstico foram evitadas para reduzir vazamento de target.

In [ ]:
feature_candidates = [
    'IDADE_ANOS',
    'DIAS_PERM',
    'QT_DIARIAS',
    'UTI_MES_IN',
    'UTI_MES_AN',
    'UTI_MES_AL',
    'UTI_MES_TO',
    'UTI_INT_IN',
    'UTI_INT_AN',
    'UTI_INT_AL',
    'UTI_INT_TO',
    'DIAR_ACOM',
    'SEXO',
    'RACA_COR',
    'GESTRISCO',
    'CONTRACEP1',
    'CONTRACEP2',
    'PROC_REA'
]

leakage_columns = [
    'CANCER_MAMA_NIVEL',
    'DIAG_PRINC',
    'DIAGSEC1',
    'DIAGSEC2',
    'DIAGSEC3',
    'DIAGSEC4',
    'DIAGSEC5',
    'DIAGSEC6',
    'DIAGSEC7',
    'CID_MORTE',
    'DIAG_PRINC_GRUPO'
]

features = [col for col in feature_candidates if col in dados.columns]
leakage_present = [col for col in leakage_columns if col in dados.columns]

print('Features selecionadas para modelagem:')
print(features)
print('\nColunas sensíveis identificadas e excluídas da modelagem:')
print(leakage_present)


In [ ]:
numerical_features = [col for col in features if dados[col].dtype.kind in 'biufc']
categorical_features = [col for col in features if col not in numerical_features]

if not numerical_features and not categorical_features:
    raise ValueError('Nenhuma feature válida foi encontrada para modelagem.')

# Limita o one-hot a colunas com cardinalidade controlada para evitar estouro de memória no Colab.
low_cardinality_features = []
high_cardinality_features = []

for col in categorical_features:
    nunique = dados[col].nunique(dropna=False)
    if nunique <= 20:
        low_cardinality_features.append(col)
    else:
        high_cardinality_features.append(col)

selected_features = numerical_features + low_cardinality_features
X = dados[selected_features].copy()
y = dados[TARGET].copy()

print('Features numéricas:', numerical_features)
print('Features categóricas de baixa cardinalidade:', low_cardinality_features)
print('Features categóricas excluídas por alta cardinalidade:', high_cardinality_features)
print('\nDimensões de X e y:', X.shape, y.shape)
print('\nValores ausentes por coluna:')
print(X.isna().sum().sort_values(ascending=False))


#### Item 3 - Separação entre treino e teste

Usamos `train_test_split` com `stratify=y` para preservar a proporção de classes entre treino e teste.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Treino:', X_train.shape, 'Teste:', X_test.shape)
print('\nProporção de classes no treino:')
print(y_train.value_counts(normalize=True).sort_index())
print('\nProporção de classes no teste:')
print(y_test.value_counts(normalize=True).sort_index())


#### Item 4 - Pipeline e treinamento dos modelos

As variáveis numéricas recebem imputação pela mediana e padronização com `StandardScaler`.
As categóricas de baixa cardinalidade recebem imputação e `OneHotEncoder`.

Serão comparados três modelos:
- `LogisticRegression` como baseline, com `class_weight='balanced'`
- `KNeighborsClassifier` como modelo principal, com ajuste simples de `k`
- `RandomForestClassifier` como alternativa baseada em árvores

Para os modelos probabilísticos, o `threshold` será definido a partir do conjunto de validação usando a curva de `precision-recall`, em vez de escolha manual.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', onehot),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, low_cardinality_features),
])

X_fit_log, X_val_log, y_fit_log, y_val_log = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

X_train_knn, _, y_train_knn, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(100_000, len(X_train)),
    random_state=42,
    stratify=y_train,
)

X_fit_knn, X_val_knn, y_fit_knn, y_val_knn = train_test_split(
    X_train_knn,
    y_train_knn,
    test_size=0.2,
    random_state=42,
    stratify=y_train_knn,
)

print('Amostra de treino para KNN:', X_train_knn.shape)

def calcular_metricas_por_threshold(y_true, y_scores):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
    f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
    rows = []
    for threshold, precision, recall, f1 in zip(thresholds, precisions[:-1], recalls[:-1], f1_scores):
        rows.append({
            'Threshold': float(threshold),
            'Precision': float(precision),
            'Recall': float(recall),
            'F1': float(f1),
        })
    return pd.DataFrame(rows)

def selecionar_threshold_por_f1(metricas_threshold):
    idx = metricas_threshold['F1'].idxmax()
    linha = metricas_threshold.loc[idx]
    return float(linha['Threshold']), float(linha['Precision']), float(linha['Recall']), float(linha['F1'])

tuning_rows = []
threshold_rows = []
model_configs = {}

logistic_validation_model = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
logistic_validation_model.fit(X_fit_log, y_fit_log)
val_scores_log = logistic_validation_model.predict_proba(X_val_log)[:, 1]
log_threshold_df = calcular_metricas_por_threshold(y_val_log, val_scores_log)
log_threshold_df['Modelo'] = 'Baseline (Logistic Regression)'
threshold_rows.extend(log_threshold_df.to_dict(orient='records'))
best_threshold_log, best_precision_log, best_recall_log, best_f1_log = selecionar_threshold_por_f1(log_threshold_df)
tuning_rows.append({
    'Modelo': 'Baseline (Logistic Regression)',
    'Parametro': f'threshold={best_threshold_log:.4f}',
    'Precision': best_precision_log,
    'Recall': best_recall_log,
    'F1': best_f1_log,
})
print(f'Melhor threshold para Logistic Regression: {best_threshold_log:.4f}')

model_configs['Baseline (Logistic Regression)'] = {
    'pipeline': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'threshold': best_threshold_log,
}
model_configs['Baseline (Logistic Regression)']['pipeline'].fit(X_train, y_train)
print('Baseline (Logistic Regression) treinado com sucesso')

best_k = None
best_knn_metrics = None
for k in [3, 5, 9, 15]:
    knn_validation_model = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', KNeighborsClassifier(n_neighbors=k)),
    ])
    knn_validation_model.fit(X_fit_knn, y_fit_knn)
    y_val_pred = knn_validation_model.predict(X_val_knn)
    precision = precision_score(y_val_knn, y_val_pred, zero_division=0)
    recall = recall_score(y_val_knn, y_val_pred, zero_division=0)
    f1 = f1_score(y_val_knn, y_val_pred, zero_division=0)
    tuning_rows.append({
        'Modelo': 'KNN',
        'Parametro': f'k={k}',
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
    })
    current_metrics = (f1, recall, precision)
    if best_knn_metrics is None or current_metrics > best_knn_metrics:
        best_knn_metrics = current_metrics
        best_k = k

print(f'Melhor valor de k para KNN: {best_k}')

model_configs['KNN'] = {
    'pipeline': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', KNeighborsClassifier(n_neighbors=best_k)),
    ]),
    'threshold': None,
}
model_configs['KNN']['pipeline'].fit(X_train_knn, y_train_knn)
print('KNN treinado com sucesso em amostra estratificada do treino')

rf_validation_model = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1,
    )),
])
rf_validation_model.fit(X_fit_log, y_fit_log)
val_scores_rf = rf_validation_model.predict_proba(X_val_log)[:, 1]
rf_threshold_df = calcular_metricas_por_threshold(y_val_log, val_scores_rf)
rf_threshold_df['Modelo'] = 'Random Forest'
threshold_rows.extend(rf_threshold_df.to_dict(orient='records'))
best_threshold_rf, best_precision_rf, best_recall_rf, best_f1_rf = selecionar_threshold_por_f1(rf_threshold_df)
tuning_rows.append({
    'Modelo': 'Random Forest',
    'Parametro': f'threshold={best_threshold_rf:.4f}',
    'Precision': best_precision_rf,
    'Recall': best_recall_rf,
    'F1': best_f1_rf,
})
print(f'Melhor threshold para Random Forest: {best_threshold_rf:.4f}')

model_configs['Random Forest'] = {
    'pipeline': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=10,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'threshold': best_threshold_rf,
}
model_configs['Random Forest']['pipeline'].fit(X_train, y_train)
print('Random Forest treinado com sucesso')

tuning_df = pd.DataFrame(tuning_rows)
threshold_df = pd.DataFrame(threshold_rows)
tuning_df.sort_values(by=['Modelo', 'F1', 'Recall'], ascending=[True, False, False]).reset_index(drop=True)


#### Item 5 - Avaliação dos modelos

Vamos calcular `accuracy`, `precision`, `recall`, `f1-score` e a matriz de confusão.
No contexto de saúde, `recall` é a métrica mais importante, pois perder casos positivos é mais custoso do que gerar falsos positivos.

A regressão logística e o Random Forest serão avaliados usando o melhor `threshold` encontrado na validação. O KNN será avaliado com o melhor `k`.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

results = []
confusion_matrices = {}

for name, config in model_configs.items():
    pipeline = config['pipeline']
    threshold = config['threshold']

    if threshold is None:
        y_pred = pipeline.predict(X_test)
    else:
        y_scores = pipeline.predict_proba(X_test)[:, 1]
        y_pred = (y_scores >= threshold).astype(int)

    matriz = confusion_matrix(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    confusion_matrices[name] = matriz

    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Configuracao': f'threshold={threshold:.2f}' if threshold is not None else f'k={best_k}',
    })

    print(f'--- {name} ---')
    if threshold is not None:
        print(f'Threshold aplicado: {threshold:.2f}')
    else:
        print(f'Valor de k aplicado: {best_k}')
    print(classification_report(y_test, y_pred, zero_division=0))
    print('Matriz de confusão:')
    print(pd.DataFrame(matriz, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1']))
    print()

results_df = pd.DataFrame(results).set_index('Modelo')
results_df = results_df.sort_values(by=['Recall', 'F1', 'Precision'], ascending=False)
results_df.round(4)


#### Item 6 - Comparação dos resultados

As tabelas abaixo resumem o ajuste dos hiperparâmetros e a comparação final dos modelos. A escolha do threshold foi feita via validação com curva de `precision-recall`.

In [ ]:
display(tuning_df.sort_values(by=['Modelo', 'F1', 'Recall'], ascending=[True, False, False]).reset_index(drop=True))

comparison = results_df[['Accuracy', 'Precision', 'Recall', 'F1', 'Configuracao']].copy()
comparison.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
}).highlight_max(subset=['Recall'], color='#d9ead3')

plt.figure(figsize=(8, 4))
tuning_plot = tuning_df.copy()
tuning_plot['Valor'] = tuning_plot['Parametro'].str.extract(r'([0-9.]+)$').astype(float)
ax = sns.lineplot(data=tuning_plot[tuning_plot['Modelo'] == 'KNN'], x='Valor', y='Recall', marker='o')
ax.set_title('Sensibilidade do KNN por valor de k')
ax.set_xlabel('k')
ax.set_ylabel('Recall')
plt.tight_layout()
plt.show()

threshold_plot_df = threshold_df.copy()
threshold_plot_df = threshold_plot_df[(threshold_plot_df['Threshold'] >= 0.0) & (threshold_plot_df['Threshold'] <= 1.0)]

logistic_threshold_reference = [0.50, 0.55, 0.58, 0.60, round(best_threshold_log, 4)]
logistic_threshold_table = threshold_plot_df[threshold_plot_df['Modelo'] == 'Baseline (Logistic Regression)'].copy()
logistic_threshold_table['ThresholdRound'] = logistic_threshold_table['Threshold'].round(4)
logistic_threshold_table = logistic_threshold_table[logistic_threshold_table['ThresholdRound'].isin(logistic_threshold_reference)]
logistic_threshold_table = logistic_threshold_table[['ThresholdRound', 'Precision', 'Recall', 'F1']].drop_duplicates().sort_values('ThresholdRound')
display(logistic_threshold_table.rename(columns={'ThresholdRound': 'Threshold'}).round(4))

plt.figure(figsize=(8, 4))
df_logistic = threshold_plot_df[threshold_plot_df['Modelo'] == 'Baseline (Logistic Regression)'].sort_values('Threshold')
sns.lineplot(data=df_logistic, x='Threshold', y='Precision', label='Precision')
sns.lineplot(data=df_logistic, x='Threshold', y='Recall', label='Recall')
sns.lineplot(data=df_logistic, x='Threshold', y='F1', label='F1')
for ref in logistic_threshold_reference:
    plt.axvline(ref, linestyle='--', color='gray', alpha=0.35)
plt.axvline(best_threshold_log, linestyle='--', color='black', alpha=0.8, label=f'Melhor threshold={best_threshold_log:.2f}')
plt.title('Trade-off das metricas por threshold - Logistic Regression')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, model_name in zip(axes, ['Baseline (Logistic Regression)', 'Random Forest']):
    df_model = threshold_plot_df[threshold_plot_df['Modelo'] == model_name].sort_values('Threshold')
    sns.lineplot(data=df_model, x='Threshold', y='Precision', ax=ax, label='Precision')
    sns.lineplot(data=df_model, x='Threshold', y='Recall', ax=ax, label='Recall')
    sns.lineplot(data=df_model, x='Threshold', y='F1', ax=ax, label='F1')
    melhor = tuning_df[tuning_df['Modelo'] == model_name].iloc[0]
    valor_threshold = float(melhor['Parametro'].split('=')[1])
    ax.axvline(valor_threshold, linestyle='--', color='black', alpha=0.6)
    ax.set_title(f'Metricas por threshold\n{model_name}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(confusion_matrices), figsize=(15, 4))
if len(confusion_matrices) == 1:
    axes = [axes]
for ax, (name, matriz) in zip(axes, confusion_matrices.items()):
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f'Matriz de confusao\n{name}')
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
    ax.set_xticklabels(['0', '1'])
    ax.set_yticklabels(['0', '1'], rotation=0)
plt.tight_layout()
plt.show()


#### Item 7 - Validação cruzada

Para alinhar a análise com o conteúdo de validação cruzada estudado na pós, vamos comparar os modelos usando `cross_val_score` com foco em `recall`.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_rows = []

cv_models = {
    'Baseline (Logistic Regression)': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'KNN': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', KNeighborsClassifier(n_neighbors=best_k)),
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=10,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

for name, model in cv_models.items():
    X_cv = X_train_knn if name == 'KNN' else X_train
    y_cv = y_train_knn if name == 'KNN' else y_train
    scores = cross_val_score(model, X_cv, y_cv, cv=cv, scoring='recall', n_jobs=1)
    cv_rows.append({
        'Modelo': name,
        'Recall CV Mean': scores.mean(),
        'Recall CV Std': scores.std(),
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values(by='Recall CV Mean', ascending=False)
cv_results_df.round(4)


#### Item 8 - ROC e AUC

Como visto na aula de ROC/AUC, modelos que retornam probabilidade podem ser comparados pela capacidade de ranquear corretamente as classes. Aqui vamos comparar `Logistic Regression` e `Random Forest`.


In [ ]:
from sklearn.metrics import auc, roc_curve

roc_rows = []
roc_curves = {}

for name in ['Baseline (Logistic Regression)', 'Random Forest']:
    pipeline = model_configs[name]['pipeline']
    y_scores = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
    roc_rows.append({
        'Modelo': name,
        'AUC': roc_auc,
    })
    print(f'--- {name} ---')
    print(f'AUC: {roc_auc:.4f}')

roc_results_df = pd.DataFrame(roc_rows).sort_values(by='AUC', ascending=False)
roc_results_df.round(4)

plt.figure(figsize=(7, 5))
for name, values in roc_curves.items():
    plt.plot(values['fpr'], values['tpr'], label=f"{name} (AUC={values['auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Aleatorio')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC dos modelos probabilisticos')
plt.legend()
plt.tight_layout()
plt.show()


#### Conclusão da Parte 6

- A regressão logística foi utilizada como baseline do grupo, com `class_weight='balanced'` para lidar melhor com o desbalanceamento.
- O `threshold` da regressão logística e do Random Forest foi ajustado com base na curva de `precision-recall` no conjunto de validação, usando o melhor `F1` em vez de valor arbitrário.
- O KNN depende diretamente da padronização das variáveis numéricas, por isso o `StandardScaler` é obrigatório.
- O valor de `k` no KNN foi ajustado em uma busca simples, priorizando `F1`, `recall` e `precision` na validação.
- O Random Forest entra como terceiro modelo para comparar uma abordagem baseada em árvores com os modelos linear e baseado em vizinhança.
- A validação cruzada foi incluída para reforçar a comparação entre modelos com uma estimativa mais estável de desempenho fora da amostra.
- A curva ROC e a AUC foram adicionadas para comparar a capacidade de ranqueamento probabilístico dos modelos binários, conforme o conteúdo estudado na aula específica de classificação.
- Para este problema, o critério principal de comparação continua sendo o `recall`, pois a prioridade é reduzir a chance de deixar passar casos com indício de câncer de mama.
- A decisão final sobre o melhor modelo deve considerar o trade-off entre `recall`, `precision` e volume de falsos positivos.